## Preparação do ambiente (execução obrigatória)

In [1]:
!pip install pandapower rich

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.1/85.1 kB 7.2 MB/s eta 0:00:00


In [ ]:
import pandapower as pp
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display


def simulate_NEW_network():
    # Criando a rede elétrica vazia
    net = pp.create_empty_network()


    # Dados fictícios para as barras (12 barras: 6 à esquerda, 6 à direita)
    barras = [
        {"id": 0, "nome": "Barra 1E", "tensao": 1.05},  # Barra de referência (Slack)
        {"id": 1, "nome": "Barra 2E", "tensao": 1.02},
        {"id": 2, "nome": "Barra 3E", "tensao": 20.0},
        {"id": 3, "nome": "Barra 4E", "tensao": 20.0},
        {"id": 4, "nome": "Barra 5E", "tensao": 20.0},
        {"id": 5, "nome": "Barra 6E", "tensao": 20.0},
        {"id": 6, "nome": "Barra 1D", "tensao": 20.0},
        {"id": 7, "nome": "Barra 2D", "tensao": 20.0},
        {"id": 8, "nome": "Barra 3D", "tensao": 20.0},
        {"id": 9, "nome": "Barra 4D", "tensao": 20.0},
        {"id": 10, "nome": "Barra 5D", "tensao": 20.0},
        {"id": 11, "nome": "Barra 6D", "tensao": 20.0},
    ]

    # Linhas conectando as barras em dois ramos (esquerdo e direito) e uma ligação central
    linhas = [
        # Lado Esquerdo (E)
        {"de": 0, "para": 1, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        {"de": 1, "para": 2, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        {"de": 2, "para": 3, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        {"de": 3, "para": 4, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        {"de": 4, "para": 5, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        # Lado Direito (D)
        {"de": 6, "para": 7, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        {"de": 7, "para": 8, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        {"de": 8, "para": 9, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        {"de": 9, "para": 10, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        {"de": 10, "para": 11, "r_ohm_per_km": 0.01, "x_ohm_per_km": 0.03, "c_nf_per_km": 10, "max_i_ka": 0.2, "comprimento_km": 1.0},
        # Ligação central entre os dois lados
        {"de": 2, "para": 8, "r_ohm_per_km": 0.02, "x_ohm_per_km": 0.04, "c_nf_per_km": 15, "max_i_ka": 0.2, "comprimento_km": 1.5},
        {"de": 5, "para": 11, "r_ohm_per_km": 0.02, "x_ohm_per_km": 0.04, "c_nf_per_km": 15, "max_i_ka": 0.2, "comprimento_km": 1.5},
    ]

    # Criando as barras
    for barra in barras:
        pp.create_bus(net, name=barra["nome"], vn_kv=barra["tensao"], index=barra["id"])

    # Criando a barra de referência (Slack)
    pp.create_ext_grid(net, bus=0, vm_pu=1.0, name="Slack")


    # Criando as linhas
    for linha in linhas:
        pp.create_line_from_parameters(
            net,
            from_bus=linha["de"],
            to_bus=linha["para"],
            length_km=linha["comprimento_km"],
            r_ohm_per_km=linha["r_ohm_per_km"],
            x_ohm_per_km=linha["x_ohm_per_km"],
            c_nf_per_km=linha["c_nf_per_km"],
            max_i_ka=linha["max_i_ka"],
            name=f"Linha {linha['de']} -> {linha['para']}",
        )

    # Dados fictícios para as cargas
    cargas = [
        {"bus": 1, "p_mw": 0.02, "q_mvar": 0.01, "nome": "Carga 1"},
        {"bus": 2, "p_mw": 0.03, "q_mvar": 0.015, "nome": "Carga 2"},
    ]

    # Criando as cargas
    for carga in cargas:
        pp.create_load(net, bus=carga["bus"], p_mw=carga["p_mw"], q_mvar=carga["q_mvar"], name=carga["nome"])

    # Dados fictícios para os geradores
    geradores = [
        {"bus": 3, "p_mw": 0.05, "vm_pu": 1.02, "nome": "Gerador PV"},
    ]

    # Criando os geradores
    for gerador in geradores:
        pp.create_sgen(net, bus=gerador["bus"], p_mw=gerador["p_mw"], vm_pu=gerador["vm_pu"], name=gerador["nome"])

    # Executando o fluxo de potência
    pp.runpp(net)

    # Exibindo os resultados
    print(f"Resultados de Sistema Elétrico de Potencia com {len(barras)} Barras:")
    print("""
        VM_PU = Tensões em pu em cada barra
        va_degree = Angulo de fase em graus
        p_mw  = Potencia Aparente em MW
        q_mvar = Potencia Reativa em MW

        + -> Potencia Demandada
        - -> Potencia Gerada
        """)
    display(net.res_bus)

    print("\nResultados das Linhas:")
    display(net.res_line)

    print("\nResultados das Cargas:")
    display(net.res_load)

    return net


def plot_results(net):
    # Plot das tensões nas barras
    fig = go.Figure()
    fig.add_trace(go.Bar(x=net.res_bus.index, y=net.res_bus.vm_pu, name="Tensão (pu)"))
    fig.update_layout(title="Tensões nas Barras", xaxis_title="Barras", yaxis_title="Tensão (pu)")
    fig.show()

    # Plot das correntes nas linhas
    fig = go.Figure()
    fig.add_trace(go.Bar(x=net.res_line.index, y=net.res_line.i_ka, name="Corrente (kA)"))
    fig.update_layout(title="Correntes nas Linhas", xaxis_title="Linhas", yaxis_title="Corrente (kA)")
    fig.show()

    # Plot das potências nas cargas
    fig = go.Figure()
    fig.add_trace(go.Bar(x=net.res_load.index, y=net.res_load.p_mw, name="Potência Ativa (MW)"))
    fig.update_layout(title="Potências Ativas nas Cargas", xaxis_title="Cargas", yaxis_title="Potência Ativa (MW)")
    fig.show()

    
    fig.to_html("./network_results.html")



def main_simulate():
    net = simulate_NEW_network()
    plot_results(net)

main_simulate()

---

## Classe (execução obrigatória)

---



In [2]:
import numpy as np
import pandapower as pp
import pandapower.networks as pw
import pandas as pd

from rich.console import Console
from rich.theme import Theme
from rich.traceback import install

install()

class Logger:
    def __init__(self):
        self.console = Console(theme=Theme({
            "success": "bold green",
            "warning": "yellow",
            "error": "bold red",
            "info": "white"  # Added "info" level for default blue color
        }))

    def log(self, message, level="info"):  # Changed default level to "info"
        """Logs a message with the specified level and color."""
        if level == "success":
            self.console.print(f"[success]{message}[/]")
        elif level == "warning":
            self.console.print(f"[warning]{message}[/]")
        elif level == "error":
            self.console.print(f"[error]{message}[/]")
        else:
            self.console.print(f"[info]{message}[/]") # Changed to "info" to use blue color



class RedeEletricaPandaPower:
    def __init__(self, network_name, debug=False):
        self.net = self.carregar_redes_padrao(network_name)
        self.debug = debug
        self.console = Logger()

        #metoodos
        self.criar_mapeamento_ramos()

        # global
        self.pesos = {
            "tensao": {"min": 100, "max": 100},
            "loading_linhas": 100,
            "loading_trafos": 150,
            "demanda": 99,
        }
        self.agendamento = pd.DataFrame()
        self.contingencia= pd.DataFrame()

    def carregar_redes_padrao(self,network_name = "14"):
        #!todo -> Switch para as redes disponiveis na lib
        match network_name:
            case "14":
                network = pw.case14()
            case "30":
                #RZ não confundir com case30
                network = pw.case_ieee30()
            case "57":
                # This function provides the ieee case57 network with the data origin PYPOWER
                network = pw.case57()
            case "118":
                network = pw.case118()

            case _:
                print("Rede não encontrada, forneça o numero como string")
                network = None

        return network

    def criar_mapeamento_ramos(self):
        """Mapeia pares de barramentos para índices de linhas e trafos"""
        self.mapeamento_ramos = {
            'linhas': {},
            'trafos': {}
        }

        # Linhas
        for idx, row in self.net.line.iterrows():
            key = tuple(sorted((row['from_bus'], row['to_bus'])))
            self.mapeamento_ramos['linhas'][key] = idx

        # Transformadores
        for idx, row in self.net.trafo.iterrows():
            key = tuple(sorted((row['hv_bus'], row['lv_bus'])))
            self.mapeamento_ramos['trafos'][key] = idx

        return self.mapeamento_ramos



    def validar_dados(self, df_agendamento, df_contingencia):
        """Valida consistência dos dados antes de processar"""
        # Verifica colunas obrigatórias
        required_agendamento = ["ramo", "inicio", "duracao", "prioridade"]
        if not all(col in df_agendamento.columns for col in required_agendamento):
            #raise ValueError("Colunas faltantes no agendamento_df")
            print("Colunas faltantes no agendamento_df")


        # Verifica existência dos ramos
        for _, row in df_agendamento.iterrows():
            ramo = tuple(sorted(row['ramo']))
            if not (ramo in self.mapeamento_ramos['linhas'] or ramo in self.mapeamento_ramos['trafos']):
                #raise ValueError(f"Ramo {row['ramo']} não existe na rede")
                print(f"Ramo {row['ramo']} não existe na rede")

        self.agendamento = df_agendamento
        self.contingencia = df_contingencia


    def hashtableindex (self, carregamento, n_carregamentos, contingencia, n_contingencias, desligamentos):
        """"
        Recebe os dados do cenário e retorna o índice da tabela hash correspondente
        carregamento -> inteiro de 1 a numero de carregamentos
        n_carregamentos -> inteiro com o número total de carregamentos
        contingencia -> inteiro de 1 a numero de contingencias
        n_contingencias -> total de contingencias
        desligamentos -> vetor linha com ndeslig elementos booleanos
        """
        num_desligamentos= len(desligamentos)

        #converte o vetor binário em inteiro de forma eficiente
        #https://stackoverflow.com/questions/24560596/fastest-way-to-convert-a-binary-listor-array-into-an-integer-in-python
        digits = ['0', '1']


        k = int("".join([ digits[y] for y in desligamentos ]), 2)

        return (k * (n_carregamentos) * (n_contingencias) ) + ((carregamento-1) * (n_contingencias))  + (contingencia-1)






    #==============================================================================================================================================================

    #! UTILS
    def log(self, mensagem,level="info"):
        if self.debug:
            self.console.log(mensagem,level)


    def show_status(self):

        if self.debug:
            print("="*80)
            print("Rede atual")
            print("="*80)

            print("\nStatus Linhas")
            display(self.net.line[["from_bus","to_bus","in_service"]])

            print("\nStatus Transformadores")
            display(self.net.trafo[["hv_bus","lv_bus","in_service"]])

            ## Barramentos
            #print("\nTensões nos Barramentos (pu):")
            #display(self.net.res_bus[['vm_pu']])

            ## linhas
            #print("\nPorcentagem de Carga nas Linhas (%):")
            #display(self.net.res_line[['loading_percent']])

            #print("\nPotência Aparente nas Linhas (MVA):")
            #display(self.net.res_line[['p_from_mw', 'q_from_mvar']])


            # transformadores
            #print("\nPotencia aparente nos transformadores")
            #display(self.net.res_trafo[['p_hv_mw', 'q_hv_mvar', 's_aparente_hv_mva', 'p_lv_mw', 'q_lv_mvar', 's_aparente_lv_mva']])


            #print("\nPorcentagem de Carga nos transformadores (%):")
            #display(self.net.res_trafo[['loading_percent']])

            #print("="*80)



    #! Otimização
    def calcular_violacoes_fitness(self):
        """
        Calcula as violações nos barramentos, linhas e transformadores.

        Considerando um peso para cada grandeza : dois pesos para tensão (max e min) e outro para loading_percent das linhas.

        Somar (valor - limite max ou limite min - valor) para calcular a aptidão daquele cenário.

        Retornar o somatório de todas as violações, mas ao soma cada violação você deve multiplicar por um peso para determinar a aptidão do cenário.

        """
        violacoes = {
            "tensao_barramentos_min": 0,
            "tensao_barramentos_max": 0,
            "loading_linhas": 0,
            "loading_trafos": 0,
        }

        # Verificar tensões nos barramentos (pu) - check
        for idx, row in self.net.res_bus.iterrows():
            # Certifique-se de que a tensão está em pu
            tensao_pu = row["vm_pu"]

            limite_max = self.net.bus.at[idx, "max_vm_pu"]
            limite_min = self.net.bus.at[idx, "min_vm_pu"]

            if tensao_pu > limite_max:
                violacoes["tensao_barramentos_max"] += tensao_pu - limite_max
            elif tensao_pu < limite_min:
                violacoes["tensao_barramentos_min"] += limite_min - tensao_pu

        # Verificar carregamento das linhas
        for idx, row in self.net.res_line.iterrows():

            carregamento = row["loading_percent"] #* 100
            limite_max = self.net.line.at[idx, "max_loading_percent"]

            if carregamento > limite_max:
                self.log("\n\nUltrapassou limite maximo nas linhas",level = "warning")
                self.log(f"{carregamento:.2f} > {limite_max} %",level = "warning")
                #RZ - as violações também devem considerar 100% = 1, deve-se dividir
                violacoes["loading_linhas"] += (carregamento - limite_max) #/ 100

        # Verificar carregamento dos transformadores
        for idx, row in self.net.res_trafo.iterrows():
            carregamento = row["loading_percent"] #* 100
            limite_max = self.net.trafo.at[idx, "max_loading_percent"]

            if carregamento > limite_max:
                self.log("\n\nUltrapassou limite maximo nos transformadores",level = "warning")
                self.log(f"{carregamento:.2f} > {limite_max} %",level = "warning")
                #RZ - as violações também devem considerar 100% = 1, deve-se dividir
                violacoes["loading_trafos"] += (carregamento - limite_max) #/ 100

        #! TODO -> PASSAR OS PESOS NA INSTANCIA DO OBJETO COM VALOR DEFAULT
        """
        Pesos das violações : (Pdem=99,Pv = 100, Pn = 100 e Pe = 150)
        onde PV é a violação de tensão max e min,
        Pdem é para não convergência do fluxo
        Pn e Pe são para o fluxo de potência (pode usar só Pn que depois explico o que é Pe).
        """


        fitness = (
            self.pesos["tensao"]["min"] * violacoes["tensao_barramentos_min"]
            + self.pesos["tensao"]["max"] * violacoes["tensao_barramentos_max"]
            + self.pesos["loading_linhas"] * violacoes["loading_linhas"]
            + self.pesos["loading_trafos"] * violacoes["loading_trafos"]
        )

        #pega as violacoes e transforma em um DF
        violacoes_df = pd.DataFrame([violacoes])

        if self.debug:
            print("\nTotal de violações e salvando num banco de dados...")
            display(violacoes_df)
            self.console.log(f"\n\nAptidão do cenário nos barramentos, linhas e transformadores ", level = "success")
            self.console.log(f"VIOLAÇÃO TOTAL  = {fitness:.2f}\n", level = "success")

        return fitness, violacoes_df


    def calcular_perfil(self,j, ls, le, ms, me, hs, he):
        """
        Calcula o perfil de carregamento (leve, médio ou pesado) para a hora `j`.

        Args:
            j (int): Hora atual.
            ls, le, ms, me, hs, he (int): Limites de horários para os perfis de carga.

        Returns:
            int: Perfil de carregamento (1 = leve, 2 = padrão, 3 = pesado).
            perfil de carga media é igual IEEE_14
            perfil de carga pesada = multiplicar todas as potencias ativas e reativas, identificando os elementos das estruturas de self.net da classe RedeEletrica do pandapower

        """
        if ls <= j % 24 < le:
            return 1  # Leve
        elif ms <= j % 24 < me:
            return 2  # Médio
        elif hs <= j % 24 < he:
            return 3  # Pesado
        return 0  # Fora dos horários definidos


    def avalia_cenarios(self, horas: int, hora_inicio: list, duracao: list, ls, le, ms, me, hs, he , debug = False):
        """
        Args:
            horas (int): Horas de duração da janela de tempo.
            hora_inicio (list): Vetor de horários iniciais dos desligamentos (valores de 0 a m-1).
            duracao (list): Vetor de duração em horas de cada desligamento.
            ls, le, ms, me, hs, he (int): Limites iniciais e finais dos horários de carregamento leve, médio e pesado.

        Returns:
            list: Matriz que armazena todos os cenários do agendamento.
        """

        matriz_cenarios = []
        num_desligamentos = len(hora_inicio)

        # Ajusta limite da janela de tempo se algum desligamento terminar fora da janela
        for i in range(num_desligamentos):
            if horas < (hora_inicio[i] + duracao[i]):
                horas = hora_inicio[i] + duracao[i]

        # Inicializa matrizes auxiliares
        matriz_desligamentos_horas = np.zeros((num_desligamentos, horas), dtype=int)
        matriz_horas = np.zeros(horas, dtype=int)

        # =================== Avaliando desligamentos por hora =================
        for j in range(horas):  # Para cada hora
            for k in range(num_desligamentos):  # Para cada desligamento
                if hora_inicio[k] <= j < (hora_inicio[k] + duracao[k]):
                    matriz_desligamentos_horas[k, j] = 1
                matriz_horas[j] += matriz_desligamentos_horas[k, j] * (2 ** k)

        # =================== Avaliando cenários =================
        for horario in range(horas):
            if horario == 0:  # Condição inicial
                if matriz_horas[horario] > 0:
                    # Se há pelo menos um desligamento ativo
                    perfil = self.calcular_perfil(horario, ls, le, ms, me, hs, he)
                    matriz_cenarios.append([perfil] + matriz_desligamentos_horas[:, horario].tolist())
            else:
                if matriz_horas[horario] != matriz_horas[horario - 1] and matriz_horas[horario] > 0:
                    # Nova topologia
                    perfil = self.calcular_perfil(horario, ls, le, ms, me, hs, he)
                    matriz_cenarios.append([perfil] + matriz_desligamentos_horas[:, horario].tolist())
                else:
                    # Mesmo cenário, mas perfil pode mudar
                    if horario - 1 in [ms, hs] and matriz_cenarios:  # Verifica se matriz_cenarios não está vazia
                        perfil = self.calcular_perfil(horario, ls, le, ms, me, hs, he)
                        matriz_cenarios[-1][0] = max(matriz_cenarios[-1][0], perfil)

        if self.debug:
            self.log("\nMatriz Cenarios:")
            for linha in matriz_cenarios:
                self.log(linha)
            self.log(f"Avaliando um total de {len(matriz_cenarios)} cenários ")


        return matriz_cenarios

    #! Pandapower New metodos
    def executar_fluxo_de_carga(self):
        """
        Executa o fluxo de carga na rede elétrica usando o algoritmo Newton-Raphson.

        Retorna:
            bool: True se o fluxo de carga convergiu, False caso contrário.
        """
        try:
            pp.runpp(self.net, algorithm="nr")
            self.log("\nFluxo de potência executado com sucesso.",level = "success")

            return True
        except pp.LoadflowNotConverged:
            self.console.log("\nErro: Fluxo de potência não convergiu.", level = "error")
            Pdem = 99

            self.calcular_violacoes_fitness()


            return False

    def ajustar_cargas(self, perfil):
        """Ajusta as cargas conforme o perfil (1 = leve, 2 = médio, 3 = pesado)."""
        tipo = ""

        if perfil == 1:
            fator = 0.941  # Carga leve

            tipo = "leve"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        elif perfil == 2:
            fator = 1.0  # Carga média (IEEE14)

            tipo = "media"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        elif perfil == 3:
            fator = 1.177  # Carga pesada

            tipo = "pesada"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        else:
            fator = 1.0  # Perfil padrão (IEEE14)

            tipo = "padrão"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        # usando o scaling
        #self.net.load["p_mw"] *= fator
        #self.net.load["q_mvar"] *= fator
        self.net.load.scaling = fator
        #self.net.gen['vm_pu'] = 1.045
        self.net.gen.scaling = fator

        self.log("Cargas ajustadas.", level = "success")

    def desligar_elementos_agendamento(self, estados):
        """Desliga os elementos (linhas e trafos) com base no cenário."""

        linhas_desligar = []
        trafos_desligar = []

        for i, estado in enumerate(estados):
            if estado == 1:  # Verifica se o ramo deve ser desligado

                ramo = self.agendamento.iloc[i]["ramo"]

                #ramo = agendamento_df.iloc[i]["ramo"]  # Obtém o ramo da tabela

                #self.log("Ramo selecionado",ramo)

                for k in range(len(self.net.line)):
                    # TODO Verifica se o ramo é uma linha ou um transformador
                    if (self.net.line["from_bus"][k] == ramo[0] and self.net.line["to_bus"][k] == ramo[1]) or (self.net.line["from_bus"][k] == ramo[1] and self.net.line["to_bus"][k] == ramo[0] ):
                        linhas_desligar.append(ramo)  # Adiciona o ramo à lista de linhas

                for k in range(len(self.net.trafo)):
                    if (self.net.trafo["hv_bus"][k] == ramo[0] and self.net.trafo["lv_bus"][k] == ramo[1]) or (self.net.trafo["hv_bus"][k] == ramo[1] and self.net.trafo["lv_bus"][k] == ramo[0] ):
                        trafos_desligar.append(ramo)  # Adiciona o ramo à lista de trafos

        self.log(f"\n\nLinhas a serem desligadas: {linhas_desligar}")
        self.log(f"Trafos a serem desligados: {trafos_desligar}\n")

        # Desliga as linhas e trafos encontrados
        self.desligar_elementos(linhas_desligar, trafos_desligar)



    def desligar_contingencia(self, ramo):
        linhas_desligar = []
        trafos_desligar = []

        self.log("Ramo selecionado",ramo)

        for k in range(len(self.net.line)):
            # TODO Verifica se o ramo é uma linha ou um transformador
            if (self.net.line["from_bus"][k] == ramo[0] and self.net.line["to_bus"][k] == ramo[1]) or (self.net.line["from_bus"][k] == ramo[1] and self.net.line["to_bus"][k] == ramo[0] ):
                    linhas_desligar.append(ramo)  # Adiciona o ramo à lista de linhas

        for k in range(len(self.net.trafo)):
            if (self.net.trafo["hv_bus"][k] == ramo[0] and self.net.trafo["lv_bus"][k] == ramo[1]) or (self.net.trafo["hv_bus"][k] == ramo[1] and self.net.trafo["lv_bus"][k] == ramo[0] ):
                    trafos_desligar.append(ramo)  # Adiciona o ramo à lista de trafos

        self.log(f"\n\nLinhas a serem desligadas: {linhas_desligar}")
        self.log(f"Trafos a serem desligados: {trafos_desligar}\n")

        # Desliga as linhas e trafos encontrados
        self.desligar_elementos(linhas_desligar, trafos_desligar)

    def desligar_elementos(self, linhas_desligar, trafos_desligar):
        # Itera pelas linhas a serem desligadas e as desliga na rede
        if not linhas_desligar:
            self.log("Nenhuma linha para desligar")

        else:
            for l in linhas_desligar:

                # Encontra o índice da linha com base em from_bus e to_bus
                index_linha = self.net.line.loc[(self.net.line['from_bus'] == l[0]) & (self.net.line['to_bus'] == l[1])].index

                # Verifica se o índice foi encontrado (CORRIGIDO AQUI PVRV)
                if not index_linha.empty:
                    # Desliga a linha usando o índice encontrado
                    self.net.line.loc[index_linha, 'in_service'] = False

                    #print(f"Linha {l} desligada com sucesso.")


                else:
                    print(f"Linha {l} não encontrada na rede.")


        # Itera pelos transformadores a serem desligados e os desliga na rede
        if not trafos_desligar:
            self.log("Nenhum transformador para desligar")
        else:
            for t in trafos_desligar:

                # Encontra o índice da linha com base em from_bus e to_bus
                index_trafo = self.net.trafo.loc[(self.net.trafo['hv_bus'] == t[0]) & (self.net.trafo['lv_bus'] == t[1])].index

                # Verifica se o índice foi encontrado
                if not index_trafo.empty:
                    # Desliga a linha usando o índice encontrado
                    self.net.trafo.loc[index_trafo, 'in_service'] = False

                    #print(f"Transformador {t} desligado com sucesso.")

                else:
                    print(f"Transformador {t} não encontrado na rede.")


    #! Old Pandapower

    #! Funções matematicas
    def calcular_potencia_aparente_trafos(self):
        """Calcula a potência aparente nos transformadores."""
        if not self.net.res_trafo.empty:
            if 's_aparente_hv_mva' not in self.net.res_trafo.columns:
                self.net.res_trafo['s_aparente_hv_mva'] = 0
            if 's_aparente_lv_mva' not in self.net.res_trafo.columns:
                self.net.res_trafo['s_aparente_lv_mva'] = 0

            # Calculando potência aparente para alta e baixa tensão
            potencia_high_tensao = (self.net.res_trafo['p_hv_mw']**2 + self.net.res_trafo['q_hv_mvar']**2)**0.5
            potencia_baixa_tensao = (self.net.res_trafo['p_lv_mw']**2 + self.net.res_trafo['q_lv_mvar']**2)**0.5

            return potencia_high_tensao, potencia_baixa_tensao
        else:
            print("Nenhum transformador na rede para calcular potência aparente.")
            return []

    def calcular_potencia_aparente_linhas(self):
        """Calcula a potência aparente nas linhas."""
        return (self.net.res_line['p_from_mw']**2 + self.net.res_line['q_from_mvar']**2)**0.5


    def religar_todos_os_ramos_agendamento(self):
        """Religa todos os ramos (linhas e transformadores) da rede elétrica."""
        # Religa todas as linhas
        self.net.line['in_service'] = True

        # Religa todos os transformadores
        self.net.trafo['in_service'] = True

        self.log("Todos os ramos religados.", level="success")



    def imprimir_resultados(self):
        """Retorna um array com todos os dados da rede elétrica"""
        dataframe = pd.DataFrame()

        # Calculo de potencia e colocando uma nova tabela no pandapower
        high_power_transformador, lower_power_transformador = self.calcular_potencia_aparente_trafos()
        fluxo_potencia_aparente_linhas = self.calcular_potencia_aparente_linhas()

        self.net.res_trafo['s_aparente_hv_mva'] = high_power_transformador
        self.net.res_trafo['s_aparente_lv_mva'] =  lower_power_transformador

        if self.debug:

            self.show_status()


        dataframe["tensao_nos_barramentos"] =  self.net.res_bus[['vm_pu']]
        dataframe["potencia_aparente_nas_linhas"] =  fluxo_potencia_aparente_linhas
        dataframe["porcentagem_de_carga_nas_linhas"] =  self.net.res_line[['loading_percent']]
        dataframe["potencia_aparente_nos_transformadores"] =  self.net.res_trafo[['p_hv_mw']]
        dataframe["porcentagem_de_carga_nos_transformadores"] =  self.net.res_trafo[['loading_percent']]

        dataframe.to_excel("dados_rede_eletrica.xlsx")
        self.log("\n\n\nDados da rede eletrica em formato de tabela excel disponivel!")

        return dataframe


## Main Function (execução opcional - teste e avaliação dos sistemas teste)
Pode ser modificada para:
- Considerar toda a main como uma função objetivo, sem debug e retornando 'fitness_final'
- Criar casos para os dados dos sistemas elétricos implementados em Zanghi(2016) - IEEE30, IEEE57, IEEE118 e SIN45
- Só executar o fluxo de potência caso a entrada de 'bd_aptidao_cenario' ainda não tenha sido calculada
- Criação da 'bd_aptidao_cenario' fora do escopo da função objetivo para permitir o acesso à base em diversas execuções do algoritmo de otimização

In [3]:
#! 1) Criar a rede elétrica IEEE 14 barras, Inicializar a classe com a rede e carrega a tabela de agendamento

casos_uso = ["14", "30", "57", "118"]

rede = RedeEletricaPandaPower(casos_uso[0], debug=False)

#! Colocando pesos como input do usuario e os dados de entrada do agendamento
rede.pesos["tensao"] = {"min": 100, "max": 100}
rede.pesos["loading_linhas"] = 100
rede.pesos["loading_trafos"] = 100

# Tabela agendamentos em xlsx hardcoded
agendamento_df = pd.DataFrame([
    {"ramo": [1, 4], "inicio": "14:00", "duracao": 6 ,"prioridade": 4},
    {"ramo": [1, 3], "inicio": "13:00", "duracao": 5, "prioridade": 1},
    {"ramo": [3, 6], "inicio": "12:00", "duracao": 6, "prioridade": 1},
    {"ramo": [11, 12], "inicio": "24:00", "duracao": 6, "prioridade": 1},
    {"ramo": [9, 10], "inicio": "18:00", "duracao": 4, "prioridade": 1}
])


contingencia_df = pd.DataFrame([
        {"contingencia":1,  "from":2 , "to": 3},
        {"contingencia":2,  "from":5 , "to": 12},
        {"contingencia":3,  "from":12 , "to": 13},
 ])

# Converter horários de início para horas do dia
agendamento_df['inicio'] = agendamento_df['inicio'].apply(lambda x: int(x.split(':')[0]))

# Calcular horário de término em horas do dia
agendamento_df['final'] = agendamento_df.apply(lambda row: (row['inicio'] + row['duracao']) % 24, axis=1)

# Calcular a duração total do agendamento em horas
duracao_total_agendamento = (agendamento_df['inicio']+agendamento_df['duracao']).max()

display(agendamento_df)

rede.validar_dados(agendamento_df, contingencia_df)




,ramo,inicio,duracao,prioridade,final
0,"[1, 4]",14,6,4,20
1,"[1, 3]",13,5,1,18
2,"[3, 6]",12,6,1,18
3,"[11, 12]",24,6,1,6
4,"[9, 10]",18,4,1,22


Recurso gráfico para impressão da rede após o fluxo de potência

Código para verificação do sistema teste e violações no fluxo de potência

In [4]:
# @title
from pandapower.plotting import simple_plot, simple_plotly, pf_res_plotly
from pandapower import diagnostic

casos_uso = ["14", "30", "57", "118"]

for i in range(len(casos_uso)):
    rede = RedeEletricaPandaPower(casos_uso[i])
    #print(rede.net.load)
    rede.ajustar_cargas(perfil=2)
    rede.executar_fluxo_de_carga()
    #print(rede.net.res_load)
    display(pf_res_plotly(rede.net))
    #diagnostic(rede.net)
    #print(rede.net.line["max_loading_percent"])
    #print(rede.net.res_line["loading_percent"])

    #print(rede.net.trafo["max_loading_percent"])
    #print(rede.net.res_trafo["loading_percent"])

    #print(rede.net.bus["min_vm_pu"])
    #print(rede.net.bus["max_vm_pu"])

    #print(rede.net.res_bus["vm_pu"])


_____________ PANDAPOWER DIAGNOSTIC TOOL _____________ 

INFO:pandapower.diagnostic_reports:Checking for missing bus indices...
INFO:pandapower.diagnostic_reports:
INFO:pandapower.diagnostic_reports:PASSED: No missing bus indices found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: No problematic switches found
 --------

INFO:pandapower.diagnostic_reports:PASSED: No connection of different voltage levels found
 --------

INFO:pandapower.diagnostic_reports:PASSED: No elements with impedance values close to zero found...
 --------

INFO:pandapower.diagnostic_reports:PASSED: No components with deviating nominal voltages found
 --------

 --------

INFO:pandapower.diagnostic_reports:PASSED: Power flow converges. No overload found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: No buses with multiple gens and/or ext_grids found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: Power flow converges. Switch configuration seems ok.
 --------

INFO:pandapower.diagnostic_re


_____________ PANDAPOWER DIAGNOSTIC TOOL _____________ 

INFO:pandapower.diagnostic_reports:Checking for missing bus indices...
INFO:pandapower.diagnostic_reports:
INFO:pandapower.diagnostic_reports:PASSED: No missing bus indices found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: No problematic switches found
 --------

INFO:pandapower.diagnostic_reports:PASSED: No connection of different voltage levels found
 --------

INFO:pandapower.diagnostic_reports:PASSED: No elements with impedance values close to zero found...
 --------

INFO:pandapower.diagnostic_reports:PASSED: No components with deviating nominal voltages found
 --------

 --------

INFO:pandapower.diagnostic_reports:PASSED: Power flow converges. No overload found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: No buses with multiple gens and/or ext_grids found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: Power flow converges. Switch configuration seems ok.
 --------

INFO:pandapower.diagnostic_re


_____________ PANDAPOWER DIAGNOSTIC TOOL _____________ 

INFO:pandapower.diagnostic_reports:Checking for missing bus indices...
INFO:pandapower.diagnostic_reports:
INFO:pandapower.diagnostic_reports:PASSED: No missing bus indices found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: No problematic switches found
 --------

INFO:pandapower.diagnostic_reports:PASSED: No connection of different voltage levels found
 --------

INFO:pandapower.diagnostic_reports:PASSED: No elements with impedance values close to zero found...
 --------

INFO:pandapower.diagnostic_reports:PASSED: No components with deviating nominal voltages found
 --------

 --------

INFO:pandapower.diagnostic_reports:PASSED: Power flow converges. No overload found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: No buses with multiple gens and/or ext_grids found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: Power flow converges. Switch configuration seems ok.
 --------

INFO:pandapower.diagnostic_re


_____________ PANDAPOWER DIAGNOSTIC TOOL _____________ 

INFO:pandapower.diagnostic_reports:Checking for missing bus indices...
INFO:pandapower.diagnostic_reports:
INFO:pandapower.diagnostic_reports:PASSED: No missing bus indices found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: No problematic switches found
 --------

INFO:pandapower.diagnostic_reports:PASSED: No connection of different voltage levels found
 --------

INFO:pandapower.diagnostic_reports:PASSED: No elements with impedance values close to zero found...
 --------

INFO:pandapower.diagnostic_reports:PASSED: No components with deviating nominal voltages found
 --------

 --------

INFO:pandapower.diagnostic_reports:PASSED: Power flow converges. No overload found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: No buses with multiple gens and/or ext_grids found.
 --------

INFO:pandapower.diagnostic_reports:PASSED: Power flow converges. Switch configuration seems ok.
 --------

INFO:pandapower.diagnostic_re

## Agendamento teste para cada caso de uso

1) Uma função objetivo para cada tipo de rede IEEE

2) Um json para cada agendamento (arquivo separado) que a função objetivo carrega, entao para os testes serão 4 arquivos separados (14, 30, 57, 118)

3) pensar na integração com o framework em DEAP com as variaveis de decisão e a função objetivo

## Função Objetivo IEEE 14

In [5]:
def funcao_objetivo_IEEE14(individuo, _debug = False):

    #! 1) Criar a rede elétrica IEEE 14 barras, Inicializar a classe com a rede e carrega a tabela de agendamento
    rede = RedeEletricaPandaPower("14", debug=_debug)

    #! Colocando pesos como input do usuario e os dados de entrada do agendamento
    rede.pesos["tensao"] = {"min": 100, "max": 100}
    rede.pesos["loading_linhas"] = 100
    rede.pesos["loading_trafos"] = 100

    # Tabela agendamentos em xlsx hardcoded
    agendamento_df = pd.DataFrame([
        {"ramo": [1, 4], "inicio": "14:00", "duracao": 6 ,"prioridade": 4},
        {"ramo": [1, 3], "inicio": "15:00", "duracao": 5, "prioridade": 1},
        {"ramo": [3, 6], "inicio": "14:00", "duracao": 6, "prioridade": 1},
        {"ramo": [11, 12], "inicio": "18:00", "duracao": 6, "prioridade": 1},
        {"ramo": [9, 10], "inicio": "15:00", "duracao": 4, "prioridade": 1}
    ])

    contingencia_df = pd.DataFrame([
            {"contingencia":1,  "from":2 , "to": 3},
            {"contingencia":2,  "from":5 , "to": 12},
            {"contingencia":3,  "from":12 , "to": 13},
    ])

    # Converter horários de início para horas do dia
    agendamento_df['inicio'] = agendamento_df['inicio'].apply(lambda x: int(x.split(':')[0]))

    # Calcular horário de término em horas do dia
    agendamento_df['final'] = agendamento_df.apply(lambda row: (row['inicio'] + row['duracao']) % 24, axis=1)

    # Calcular a duração total do agendamento em horas
    duracao_total_agendamento = (agendamento_df['inicio']+agendamento_df['duracao']).max()
    rede.validar_dados(agendamento_df, contingencia_df)

    # passando a variavel de decisão na função objetivo
    agendamento_df["inicio"] = individuo


    #=====================================================

    # 2)  Avaliar cenários e criar matriz de cenários
    matriz_cenarios = rede.avalia_cenarios(
            horas = duracao_total_agendamento,
            hora_inicio=agendamento_df['inicio'],
            duracao=agendamento_df['duracao'],
            ls=0, le=8,
            ms=8, me=18,
            hs=18, he=24
        )

    #! Calculo  de otimização para achar o fitness de cada cenario
    violacoes_total = []
    violacoes_hash_table = {}

    # Generate hash key (teste 01)
    contingencias = contingencia_df['contingencia'].to_list()
    num_carregamentos = 3
    num_contingencias = len(contingencias) # 3
    num_desligamentos = len(agendamento_df) # 5

    # FAZENDO UM BANCO EM MEMORIA DE EXECUÇÃO
    bd_aptidao_cenario =[-1.0]*(num_contingencias* num_carregamentos*(2**num_desligamentos) )

    try:
        # 3) Processar cada cenário da matriz de cenários
        for cenario in matriz_cenarios:
            perfil = cenario[0]
            estado_ramos = cenario[1:]

            # 4) Ajustar carregamento para o perfil do cenário
            rede.ajustar_cargas(perfil)

            # Loop through contingencies before calculating violations for the scenario
            for contingencia_atual in range(num_contingencias):
                contingencia_atual += 1

                #5)  Ligar todos os ramos antes de aplicar mudanças
                rede.religar_todos_os_ramos_agendamento()

                # 6) Fazendo os deligamentos com base na tabela em .xlsx e nos cenários calculados
                rede.desligar_elementos_agendamento(estado_ramos)

                # 7) Identifica ramos afetados pela contingência
                ramo_contingencia = list(contingencia_df.loc[contingencia_df['contingencia'] == contingencia_atual, ['from', 'to']].values[0])
                rede.log(f"\n{contingencia_atual}) Ramo da contingencia = { ramo_contingencia}\n")

                # 8) Desliga os ramos afetados
                rede.desligar_contingencia(ramo_contingencia)

                # 9) Executar fluxo de potência para o cenário com contingência
                if rede.executar_fluxo_de_carga():

                    # 10) Calcular violações com pesos e armazenar os resultados
                    fitness, violacoes_df = rede.calcular_violacoes_fitness()
                    violacoes_total.append(fitness)

                else:
                    fitness = rede.pesos["demanda"] # penalidade com valor default de 99

                # 11) Store violation in the hash table
                hash_key = rede.hashtableindex(perfil, num_carregamentos, contingencia_atual, num_contingencias, estado_ramos)

                violacoes_hash_table[hash_key] = fitness

                bd_aptidao_cenario[hash_key] = fitness
                rede.log(f"Hash key = { hash_key}\n")


            #! Ver apenas o true in service de barras e transformadores
            rede.show_status()

        #! Usando dicionario nos temos os valores acumulando tirando os valores nulos
        hash_df2 = pd.DataFrame(violacoes_hash_table.items(), columns=['Hash Key', 'Fitness'])

        # Passando os valores do array direto no dataframe com os index como chave (hash = chave, valor)
        hash_df = pd.DataFrame(bd_aptidao_cenario, columns=[ 'Fitness'])
        filtered_hash_table = hash_df.loc[hash_df['Fitness'] > 0]

        hash_df.to_excel("hash_table.xlsx", index=False)

        # 12) Calcular fitness final com somatorio das vioações com pesos de todos os cenarios
        fitness_final = sum(violacoes_total)
        rede.log(f"\nFitness do agendamento = {fitness_final:.2f}\n")
        return fitness_final

    except Exception as e:
        print(f"\nErro: {e}")

In [6]:
funcao_objetivo_IEEE14(
    #agendamento proposto em Zanghi(2016)
    individuo=[14,15,14,18,15],

    #agendamento ótimo em Zanghi(2016)
    #individuo=[14,13,12,24,18],
    _debug = True
)


Matriz Cenarios:

[2, 1, 0, 1, 0, 0]

[2, 1, 1, 1, 0, 1]

[3, 1, 1, 1, 1, 1]

[3, 1, 1, 1, 1, 0]

...

,from_bus,to_bus,in_service
0,0,1,True
1,0,4,True
2,1,2,True
3,1,3,True
4,1,4,True
5,2,3,True
6,3,4,True
7,5,10,True
8,5,11,True
9,5,12,True



Status Transformadores


,hv_bus,lv_bus,in_service
0,3,6,True
1,3,8,True
2,4,5,True
3,6,7,True
4,6,8,True


Fitness do agendamento = 114.43

np.float64(114.42965594038847)

## Função Objetivo IEEE 30

In [7]:
def funcao_objetivo_IEEE30(individuo, _debug = False):
    try:

        #! 1) Criar a rede elétrica IEEE 14 barras, Inicializar a classe com a rede e carrega a tabela de agendamento
        rede = RedeEletricaPandaPower("30", debug=_debug)

        #rede.show_status()

        #! Colocando pesos como input do usuario e os dados de entrada do agendamento
        rede.pesos["tensao"] = {"min": 100, "max": 100}
        rede.pesos["loading_linhas"] = 100
        rede.pesos["loading_trafos"] = 100

        #! Tabela agendamentos em xlsx hardcoded
        agendamento_df = pd.DataFrame([
            {"ramo": [1, 3], "inicio": "15:00", "duracao": 6 ,"prioridade": 4},
            {"ramo": [1, 5], "inicio": "15:00", "duracao": 5, "prioridade": 1},
            {"ramo": [5, 8], "inicio": "14:00", "duracao": 6, "prioridade": 1},
            {"ramo": [13, 14], "inicio": "18:00", "duracao": 6, "prioridade": 1},
            {"ramo": [15, 16], "inicio": "15:00", "duracao": 4, "prioridade": 1},
            {"ramo": [21, 23], "inicio": "14:00", "duracao": 5, "prioridade": 1},
            {"ramo": [7, 27], "inicio": "10:00", "duracao": 6, "prioridade": 1},
            {"ramo": [26, 28], "inicio": "14:00", "duracao": 5, "prioridade": 1},
            {"ramo": [9, 21], "inicio": "18:00", "duracao": 4, "prioridade": 1},
            {"ramo": [14, 17], "inicio": "15:00", "duracao": 5, "prioridade": 1},

        ])

        contingencia_df = pd.DataFrame([
                {"contingencia":1,  "from":1 , "to": 3},
                {"contingencia":2,  "from":11 , "to": 14},
                {"contingencia":3,  "from":14 , "to": 17},
        ])

        # Converter horários de início para horas do dia
        agendamento_df['inicio'] = agendamento_df['inicio'].apply(lambda x: int(x.split(':')[0]))

        # Calcular horário de término em horas do dia
        agendamento_df['final'] = agendamento_df.apply(lambda row: (row['inicio'] + row['duracao']) % 24, axis=1)

        # Calcular a duração total do agendamento em horas
        duracao_total_agendamento = (agendamento_df['inicio']+agendamento_df['duracao']).max()
        rede.validar_dados(agendamento_df, contingencia_df)

        # passando a variavel de decisão na função objetivo
        agendamento_df["inicio"] = individuo



        #=====================================================

        # 2)  Avaliar cenários e criar matriz de cenários
        matriz_cenarios = rede.avalia_cenarios(
                horas = duracao_total_agendamento,
                hora_inicio=agendamento_df['inicio'],
                duracao=agendamento_df['duracao'],
                ls=0, le=8,
                ms=8, me=18,
                hs=18, he=24
            )

        #! Calculo  de otimização para achar o fitness de cada cenario
        violacoes_total = []
        violacoes_hash_table = {}

        # Generate hash key (teste 01)
        contingencias = contingencia_df['contingencia'].to_list()
        num_carregamentos = 3
        num_contingencias = len(contingencias) # 3
        num_desligamentos = len(agendamento_df) # 5

        # FAZENDO UM BANCO EM MEMORIA DE EXECUÇÃO
        bd_aptidao_cenario =[-1.0]*(num_contingencias* num_carregamentos*(2**num_desligamentos) )


        # 3) Processar cada cenário da matriz de cenários
        for cenario in matriz_cenarios:
            perfil = cenario[0]
            estado_ramos = cenario[1:]


            # 4) Ajustar carregamento para o perfil do cenário
            rede.ajustar_cargas(perfil)


            # Loop through contingencies before calculating violations for the scenario
            for contingencia_atual in range(num_contingencias):
                contingencia_atual += 1

                #5)  Ligar todos os ramos antes de aplicar mudanças
                rede.religar_todos_os_ramos_agendamento()

                # 6) Fazendo os deligamentos com base na tabela em .xlsx e nos cenários calculados
                rede.desligar_elementos_agendamento(estado_ramos)

                # 7) Identifica ramos afetados pela contingência
                ramo_contingencia = list(contingencia_df.loc[contingencia_df['contingencia'] == contingencia_atual, ['from', 'to']].values[0])
                rede.log(f"\n{contingencia_atual}) Ramo da contingencia = { ramo_contingencia}\n")

                # 8) Desliga os ramos afetados
                rede.desligar_contingencia(ramo_contingencia)

                # 9) Executar fluxo de potência para o cenário com contingência
                if rede.executar_fluxo_de_carga():

                    # 10) Calcular violações com pesos e armazenar os resultados
                    fitness, violacoes_df = rede.calcular_violacoes_fitness()
                    violacoes_total.append(fitness)

                else:
                    fitness = rede.pesos["demanda"] # penalidade com valor default de 99

                # 11) Store violation in the hash table
                hash_key = rede.hashtableindex(perfil, num_carregamentos, contingencia_atual, num_contingencias, estado_ramos)

                violacoes_hash_table[hash_key] = fitness

                bd_aptidao_cenario[hash_key] = fitness
                rede.log(f"Hash key = { hash_key}\n")


            #! Ver apenas o true in service de barras e transformadores
            rede.show_status()

        #! Usando dicionario nos temos os valores acumulando tirando os valores nulos
        hash_df2 = pd.DataFrame(violacoes_hash_table.items(), columns=['Hash Key', 'Fitness'])

        # Passando os valores do array direto no dataframe com os index como chave (hash = chave, valor)
        hash_df = pd.DataFrame(bd_aptidao_cenario, columns=[ 'Fitness'])
        filtered_hash_table = hash_df.loc[hash_df['Fitness'] > 0]

        hash_df.to_excel("hash_table.xlsx", index=False)

        # 12) Calcular fitness final com somatorio das vioações com pesos de todos os cenarios
        fitness_final = sum(violacoes_total)
        rede.log(f"\nFitness do agendamento = {fitness_final:.2f}\n")
        return fitness_final


    except Exception as e:
        print(f"\nErro: {e}")

In [8]:
funcao_objetivo_IEEE30(
    #agendamento proposto em Zanghi(2016)
    individuo=[15,15,14,18,15,14,10,14,18,15],
    #agendamento ótimo em Zanghi(2016)
    #individuo=[15,15,10,21,16,13,10,14,17,18],
    _debug = False
)

Erro: Fluxo de potência não convergiu.

np.float64(148.58480710566118)

## Função objetivo IEEE 57

In [9]:
def funcao_objetivo_IEEE57(individuo, _debug = False):

    #! 1) Criar a rede elétrica IEEE 14 barras, Inicializar a classe com a rede e carrega a tabela de agendamento
    rede = RedeEletricaPandaPower("57", debug=_debug)

    #! Colocando pesos como input do usuario e os dados de entrada do agendamento
    rede.pesos["tensao"] = {"min": 100, "max": 100}
    rede.pesos["loading_linhas"] = 100
    rede.pesos["loading_trafos"] = 100

    #! Tabela agendamentos em xlsx hardcoded
    agendamento_df = pd.DataFrame([
        {"ramo": [2, 3], "inicio": "08:00", "duracao": 6 ,"prioridade": 4},
        {"ramo": [8, 10], "inicio": "10:00", "duracao": 5, "prioridade": 1},
        {"ramo": [25, 26], "inicio": "14:00", "duracao": 6, "prioridade": 1},
        {"ramo": [12, 14], "inicio": "18:00", "duracao": 6, "prioridade": 1},
        {"ramo": [10, 40], "inicio": "15:00", "duracao": 4, "prioridade": 1},
        {"ramo": [37, 48], "inicio": "08:00", "duracao": 5, "prioridade": 1},
        {"ramo": [51, 52], "inicio": "10:00", "duracao": 6, "prioridade": 1},
        {"ramo": [39, 55], "inicio": "14:00", "duracao": 5, "prioridade": 1},
        {"ramo": [45, 46], "inicio": "18:00", "duracao": 4, "prioridade": 1},
        {"ramo": [17, 18], "inicio": "15:00", "duracao": 5, "prioridade": 1},

    ])

    contingencia_df = pd.DataFrame([
            {"contingencia":1,  "from":1 , "to": 2},
            {"contingencia":2,  "from":8 , "to": 9},
            {"contingencia":3,  "from":43 , "to": 44},
    ])

    # Converter horários de início para horas do dia
    agendamento_df['inicio'] = agendamento_df['inicio'].apply(lambda x: int(x.split(':')[0]))

    # Calcular horário de término em horas do dia
    agendamento_df['final'] = agendamento_df.apply(lambda row: (row['inicio'] + row['duracao']) % 24, axis=1)

    # Calcular a duração total do agendamento em horas
    duracao_total_agendamento = (agendamento_df['inicio']+agendamento_df['duracao']).max()
    rede.validar_dados(agendamento_df, contingencia_df)

    # passando a variavel de decisão na função objetivo
    agendamento_df["inicio"] = individuo



    #=====================================================

    # 2)  Avaliar cenários e criar matriz de cenários
    matriz_cenarios = rede.avalia_cenarios(
            horas = duracao_total_agendamento,
            hora_inicio=agendamento_df['inicio'],
            duracao=agendamento_df['duracao'],
            ls=0, le=8,
            ms=8, me=18,
            hs=18, he=24
        )

    #! Calculo  de otimização para achar o fitness de cada cenario
    violacoes_total = []
    violacoes_hash_table = {}

    # Generate hash key (teste 01)
    contingencias = contingencia_df['contingencia'].to_list()
    num_carregamentos = 3
    num_contingencias = len(contingencias) # 3
    num_desligamentos = len(agendamento_df) # 5

    # FAZENDO UM BANCO EM MEMORIA DE EXECUÇÃO
    bd_aptidao_cenario =[-1.0]*(num_contingencias* num_carregamentos*(2**num_desligamentos) )

    try:
        # 3) Processar cada cenário da matriz de cenários
        for cenario in matriz_cenarios:
            perfil = cenario[0]
            estado_ramos = cenario[1:]


            # 4) Ajustar carregamento para o perfil do cenário
            rede.ajustar_cargas(perfil)


            # Loop through contingencies before calculating violations for the scenario
            for contingencia_atual in range(num_contingencias):
                contingencia_atual += 1

                #5)  Ligar todos os ramos antes de aplicar mudanças
                rede.religar_todos_os_ramos_agendamento()

                # 6) Fazendo os deligamentos com base na tabela em .xlsx e nos cenários calculados
                rede.desligar_elementos_agendamento(estado_ramos)

                # 7) Identifica ramos afetados pela contingência
                ramo_contingencia = list(contingencia_df.loc[contingencia_df['contingencia'] == contingencia_atual, ['from', 'to']].values[0])
                rede.log(f"\n{contingencia_atual}) Ramo da contingencia = { ramo_contingencia}\n")

                # 8) Desliga os ramos afetados
                rede.desligar_contingencia(ramo_contingencia)

                # 9) Executar fluxo de potência para o cenário com contingência
                if rede.executar_fluxo_de_carga():

                    # 10) Calcular violações com pesos e armazenar os resultados
                    fitness, violacoes_df = rede.calcular_violacoes_fitness()
                    violacoes_total.append(fitness)

                else:
                    fitness = rede.pesos["demanda"] # penalidade com valor default de 99

                # 11) Store violation in the hash table
                hash_key = rede.hashtableindex(perfil, num_carregamentos, contingencia_atual, num_contingencias, estado_ramos)

                violacoes_hash_table[hash_key] = fitness

                bd_aptidao_cenario[hash_key] = fitness
                rede.log(f"Hash key = { hash_key}\n")


            #! Ver apenas o true in service de barras e transformadores
            rede.show_status()

        #! Usando dicionario nos temos os valores acumulando tirando os valores nulos
        hash_df2 = pd.DataFrame(violacoes_hash_table.items(), columns=['Hash Key', 'Fitness'])

        # Passando os valores do array direto no dataframe com os index como chave (hash = chave, valor)
        hash_df = pd.DataFrame(bd_aptidao_cenario, columns=[ 'Fitness'])
        filtered_hash_table = hash_df.loc[hash_df['Fitness'] > 0]

        hash_df.to_excel("hash_table.xlsx", index=False)

        # 12) Calcular fitness final com somatorio das vioações com pesos de todos os cenarios
        fitness_final = sum(violacoes_total)
        rede.log(f"\nFitness do agendamento = {fitness_final:.2f}\n")
        return fitness_final


    except Exception as e:
        print(f"\nErro: {e}")

In [10]:
funcao_objetivo_IEEE57(
    #agendamento proposto em Zanghi(2016)
    individuo=[8,10,14,18,15,8,10,14,18,15],
    #agendamento ótimo em Zanghi(2016)
    #individuo=[8,10,28,24,14,3,10,13,24,13],
    _debug = True
)

Matriz Cenarios:

[2, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0]

[2, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0]

[2, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0]

[2, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0]

...

,from_bus,to_bus,in_service
0,0,1,True
1,1,2,True
2,2,3,True
3,3,4,True
4,3,5,True
...,...,...,...
58,55,40,True
59,55,41,True
60,56,55,True
61,37,48,True



Status Transformadores


,hv_bus,lv_bus,in_service
0,17,3,True
1,17,3,True
2,19,20,True
3,24,23,True
4,24,23,True
5,25,23,True
6,28,6,True
7,31,33,True
8,40,10,True
9,44,14,True


Fitness do agendamento = 11410.02

np.float64(11410.01641113137)

## Função Objetivo IEEE 118

In [11]:
def funcao_objetivo_IEEE118(individuo, _debug = False):

    #! 1) Criar a rede elétrica IEEE 14 barras, Inicializar a classe com a rede e carrega a tabela de agendamento
    rede = RedeEletricaPandaPower("118", debug=_debug)

    #! Colocando pesos como input do usuario e os dados de entrada do agendamento
    rede.pesos["tensao"] = {"min": 100, "max": 100}
    rede.pesos["loading_linhas"] = 100
    rede.pesos["loading_trafos"] = 100

    #! Tabela agendamentos em xlsx hardcoded
    agendamento_df = pd.DataFrame([
        {"ramo": [7, 29], "inicio": "20:00", "duracao": 6 ,"prioridade": 4},
        {"ramo": [44, 48], "inicio": "18:00", "duracao": 5, "prioridade": 1},
        {"ramo": [16, 112], "inicio": "21:00", "duracao": 6, "prioridade": 1},
        {"ramo": [61, 65], "inicio": "27:00", "duracao": 6, "prioridade": 1}, # dia seguinte
        {"ramo": [75, 117], "inicio": "01:00", "duracao": 4, "prioridade": 1},
        {"ramo": [46, 68], "inicio": "21:00", "duracao": 5, "prioridade": 1},
        {"ramo": [84, 88], "inicio": "20:00", "duracao": 6, "prioridade": 1},
        {"ramo": [18, 33], "inicio": "14:00", "duracao": 5, "prioridade": 1},
        {"ramo": [3, 10], "inicio": "19:00", "duracao": 4, "prioridade": 1},
        {"ramo": [11, 15], "inicio": "20:00", "duracao": 5, "prioridade": 1},

    ])

    contingencia_df = pd.DataFrame([
            {"contingencia":1,  "from":48 , "to": 49},
            {"contingencia":2,  "from":10 , "to": 11},
            {"contingencia":3,  "from":16 , "to": 17},
    ])

    # Converter horários de início para horas do dia
    agendamento_df['inicio'] = agendamento_df['inicio'].apply(lambda x: int(x.split(':')[0]))

    # Calcular horário de término em horas do dia
    agendamento_df['final'] = agendamento_df.apply(lambda row: (row['inicio'] + row['duracao']) % 24, axis=1)

    # Calcular a duração total do agendamento em horas
    duracao_total_agendamento = (agendamento_df['inicio']+agendamento_df['duracao']).max()
    rede.validar_dados(agendamento_df, contingencia_df)

    # passando a variavel de decisão na função objetivo
    agendamento_df["inicio"] = individuo



    #=====================================================

    # 2)  Avaliar cenários e criar matriz de cenários
    matriz_cenarios = rede.avalia_cenarios(
            horas = duracao_total_agendamento,
            hora_inicio=agendamento_df['inicio'],
            duracao=agendamento_df['duracao'],
            ls=0, le=8,
            ms=8, me=18,
            hs=18, he=24
        )

    #! Calculo  de otimização para achar o fitness de cada cenario
    violacoes_total = []
    violacoes_hash_table = {}

    # Generate hash key (teste 01)
    contingencias = contingencia_df['contingencia'].to_list()
    num_carregamentos = 3
    num_contingencias = len(contingencias) # 3
    num_desligamentos = len(agendamento_df) # 5

    # FAZENDO UM BANCO EM MEMORIA DE EXECUÇÃO
    bd_aptidao_cenario =[-1.0]*(num_contingencias* num_carregamentos*(2**num_desligamentos) )

    try:
        # 3) Processar cada cenário da matriz de cenários
        for cenario in matriz_cenarios:
            perfil = cenario[0]
            estado_ramos = cenario[1:]


            # 4) Ajustar carregamento para o perfil do cenário
            rede.ajustar_cargas(perfil)


            # Loop through contingencies before calculating violations for the scenario
            for contingencia_atual in range(num_contingencias):
                contingencia_atual += 1

                #5)  Ligar todos os ramos antes de aplicar mudanças
                rede.religar_todos_os_ramos_agendamento()

                # 6) Fazendo os deligamentos com base na tabela em .xlsx e nos cenários calculados
                rede.desligar_elementos_agendamento(estado_ramos)

                # 7) Identifica ramos afetados pela contingência
                ramo_contingencia = list(contingencia_df.loc[contingencia_df['contingencia'] == contingencia_atual, ['from', 'to']].values[0])
                rede.log(f"\n{contingencia_atual}) Ramo da contingencia = { ramo_contingencia}\n")

                # 8) Desliga os ramos afetados
                rede.desligar_contingencia(ramo_contingencia)

                # 9) Executar fluxo de potência para o cenário com contingência
                if rede.executar_fluxo_de_carga():

                    # 10) Calcular violações com pesos e armazenar os resultados
                    fitness, violacoes_df = rede.calcular_violacoes_fitness()
                    violacoes_total.append(fitness)

                else:
                    fitness = rede.pesos["demanda"] # penalidade com valor default de 99

                # 11) Store violation in the hash table
                hash_key = rede.hashtableindex(perfil, num_carregamentos, contingencia_atual, num_contingencias, estado_ramos)

                violacoes_hash_table[hash_key] = fitness

                bd_aptidao_cenario[hash_key] = fitness
                rede.log(f"Hash key = { hash_key}\n")


            #! Ver apenas o true in service de barras e transformadores
            rede.show_status()

        #! Usando dicionario nos temos os valores acumulando tirando os valores nulos
        hash_df2 = pd.DataFrame(violacoes_hash_table.items(), columns=['Hash Key', 'Fitness'])

        # Passando os valores do array direto no dataframe com os index como chave (hash = chave, valor)
        hash_df = pd.DataFrame(bd_aptidao_cenario, columns=[ 'Fitness'])
        filtered_hash_table = hash_df.loc[hash_df['Fitness'] > 0]

        hash_df.to_excel("hash_table.xlsx", index=False)

        # 12) Calcular fitness final com somatorio das vioações com pesos de todos os cenarios
        fitness_final = sum(violacoes_total)
        rede.log(f"\nFitness do agendamento = {fitness_final:.2f}\n")


        return fitness_final


    except Exception as e:
        print(f"\nErro: {e}")

In [12]:
funcao_objetivo_IEEE118(
    #agendamento proposto em Zanghi(2016)
    #individuo=[20,18,21,27,1,21,20,14,19,20],

    #agendamento ótimo em Zanghi(2016)
    individuo=[24,3,24,26,1,24,24,27,24,24],
    _debug = True
)

Matriz Cenarios:

[1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]

[1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0]

[3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]

[1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1]

...

,from_bus,to_bus,in_service
0,0,1,True
1,0,2,True
2,3,4,True
3,2,4,True
4,4,5,True
...,...,...,...
168,26,114,True
169,113,114,True
170,11,116,True
171,74,117,True



Status Transformadores


,hv_bus,lv_bus,in_service
0,7,4,True
1,25,24,True
2,29,16,True
3,37,36,True
4,62,58,True
5,63,60,True
6,64,65,True
7,64,67,True
8,67,68,True
9,80,67,True


Fitness do agendamento = 11.44

np.float64(11.438722723686201)